# Notebook 2.1: The Wigner Quasiprobability Distribution Fundamentals
### Module 2: Phase Space and Quasiprobability Distributions
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_2_phase_space_quasiprobabilities/01_wigner_function_fundamentals.ipynb)

---

### Prerequisites
- Module 1: Single-Mode Quantum States of Light (Fock, Coherent, Squeezed, and Thermal States)
- Notebook 0.3: Observables, Measurement, and Uncertainty Principles
- Notebook 0.4: The Quantum Harmonic Oscillator

### Pedagogical Learning Objectives
1. **The Quantum Phase Space Dilemma:** Understand why the Heisenberg uncertainty principle $[\hat{x}, \hat{p}] = i\hbar$ forbids a true joint probability distribution $P(x, p)$, motivating Eugene Wigner's 1932 quasiprobability construction.
2. **Mathematical Definition & Symmetries:** Formulate $W(x, p)$ in position wavefunction and density operator representations, establishing reality ($W \in \mathbb{R}$) and unit normalization ($\iint W \, dx \, dp = 1$).
3. **The Marginal Distributions Theorem:** Prove and numerically verify that integrating $W(x, p)$ along any direction in phase space yields the exact quantum probability distribution of that quadrature.
4. **Royer's Displaced Parity Operator:** Express $W(x, p)$ via the expectation value of the displaced parity operator $\hat{\Pi}(\alpha) = \hat{D}(\alpha)(-1)^{\hat{n}}\hat{D}^\dagger(\alpha)$, proving that $W(0, 0) = \frac{1}{\pi}\langle (-1)^{\hat{n}} \rangle$.
5. **The Moyal Overlap Formula:** Calculate quantum state overlaps and purity directly in phase space via $\mathrm{Tr}(\hat{\rho}_1 \hat{\rho}_2) = 2\pi \iint W_1(x, p) W_2(x, p) \, dx \, dp$.
6. **Numerical Synthesis & Visual Exploration:** Implement a generalized Fock-basis Laguerre solver to render publication-quality 2D contour maps and 3D surface landscapes for canonical quantum states.


In [ ]:
# Setup, environment detection, and library imports
import os
import sys
import math
import numpy as np
import scipy.linalg as la
from scipy.special import genlaguerre
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Ensure matplotlib handles headless environments gracefully
if 'MPLBACKEND' not in os.environ:
    os.environ['MPLBACKEND'] = 'Agg' if not hasattr(sys, 'ps1') else 'module://matplotlib_inline.backend_inline'

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, FloatSlider, Dropdown
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. The Quantum Phase Space Dilemma and Wigner's 1932 Formulation

In classical Hamiltonian mechanics, the state of a particle is fully specified by a point $(x, p)$ in phase space, and an ensemble is described by a positive probability density $P_{\text{cl}}(x, p) \ge 0$.

In quantum mechanics, however, position $\hat{x}$ and momentum $\hat{p}$ are non-commuting operators:
$$
[\hat{x}, \hat{p}] = i\hbar
$$
By the Heisenberg uncertainty principle, **a quantum particle cannot simultaneously possess well-defined position and momentum**. Consequently, a true joint probability distribution $P(x, p)$ does not exist!

---

### Eugene Wigner's 1932 Quasiprobability Distribution

In 1932, **Eugene Wigner** introduced a real-valued phase-space distribution to study quantum corrections to classical thermodynamics (*Physical Review* 40, 749, 1932).

For a pure state with position wavefunction $\psi(x)$:

$$
W(x, p) \equiv \frac{1}{\pi \hbar} \int_{-\infty}^\infty \psi^*(x + y) \psi(x - y) e^{2ipy/\hbar} \, dy
$$

For an arbitrary mixed state described by density matrix $\hat{\rho}$:

$$
W(x, p) \equiv \frac{1}{\pi \hbar} \int_{-\infty}^\infty \langle x + y | \hat{\rho} | x - y \rangle e^{-2ipy/\hbar} \, dy
$$

In natural quantum optics units where $\hbar = 1$ and quadratures are dimensionless ($\hat{x} = \frac{\hat{a}+\hat{a}^\dagger}{\sqrt{2}}$, $\hat{p} = \frac{\hat{a}-\hat{a}^\dagger}{i\sqrt{2}}$), the prefactor becomes $\frac{1}{\pi}$.

---

### Fundamental Mathematical Properties of $W(x, p)$

1. **Strict Reality:**
   $$
   W(x, p) = W^*(x, p) \in \mathbb{R} \quad \text{for all } (x, p)
   $$
2. **Total Unit Normalization:**
   $$
   \iint_{-\infty}^\infty W(x, p) \, dx \, dp = \mathrm{Tr}(\hat{\rho}) = 1
   $$
3. **Exact Marginal Distributions:**
   Integrating out momentum yields the exact spatial probability density:
   $$
   \int_{-\infty}^\infty W(x, p) \, dp = \langle x | \hat{\rho} | x \rangle = |\psi(x)|^2
   $$
   Integrating out position yields the exact momentum probability density:
   $$
   \int_{-\infty}^\infty W(x, p) \, dx = \langle p | \hat{\rho} | p \rangle = |\tilde{\psi}(p)|^2
   $$
4. **Moyal Overlap Formula:**
   The transition probability or state fidelity between two states $\hat{\rho}_1$ and $\hat{\rho}_2$ is given by:
   $$
   \mathrm{Tr}(\hat{\rho}_1 \hat{\rho}_2) = 2\pi \iint_{-\infty}^\infty W_1(x, p) W_2(x, p) \, dx \, dp
   $$
   Setting $\hat{\rho}_1 = \hat{\rho}_2 = \hat{\rho}$ yields the **purity**:
   $$
   \mathcal{P} = \mathrm{Tr}(\hat{\rho}^2) = 2\pi \iint_{-\infty}^\infty [W(x, p)]^2 \, dx \, dp \le 1
   $$


## 2. Royer's Parity Formulation & Analytical Fock Distributions

### Antoine Royer's 1977 Theorem

In 1977, Antoine Royer established a profound physical interpretation of the Wigner function (*Physical Review A* 15, 449, 1977): **the Wigner function at phase-space coordinate $\alpha = \frac{x + ip}{\sqrt{2}}$ is proportional to the expectation value of the displaced parity operator!**

$$
W(x, p) = \frac{1}{\pi} \mathrm{Tr}\left[ \hat{\rho} \, \hat{\Pi}(x, p) \right]
$$

where $\hat{\Pi}(x, p)$ is the **displaced parity operator**:
$$
\hat{\Pi}(x, p) = \hat{D}(\alpha) (-1)^{\hat{n}} \hat{D}^\dagger(\alpha), \qquad \alpha = \frac{x + ip}{\sqrt{2}}
$$
and $(-1)^{\hat{n}} = \sum_{n=0}^\infty (-1)^n |n\rangle\langle n|$ is the photon parity operator.

---

### The Origin Parity Value $W(0, 0)$

Setting $\alpha = 0$ ($\hat{D}(0) = \hat{I}$) yields a direct, elegant relationship:
$$
W(0, 0) = \frac{1}{\pi} \langle (-1)^{\hat{n}} \rangle = \frac{1}{\pi} \sum_{n=0}^\infty (-1)^n P_n = \frac{1}{\pi} \left( P_{\text{even}} - P_{\text{odd}} \right)
$$
- For the vacuum state $|0\rangle$: $P_0 = 1 \Longrightarrow W(0, 0) = +\frac{1}{\pi} \approx +0.3183$.
- For the single-photon Fock state $|1\rangle$: $P_1 = 1 \Longrightarrow W(0, 0) = -\frac{1}{\pi} \approx -0.3183 < 0$!

This negative value is why $W(x, p)$ is termed a **quasiprobability** distribution: while its marginals are positive real probabilities, $W(x, p)$ itself can take **negative values**, which serve as an incontrovertible signature of non-classical quantum interference!

---

### Analytical Formula for Fock States

For an exact photon number state $|n\rangle$, the Wigner function evaluates analytically to:

$$
W_n(x, p) = \frac{(-1)^n}{\pi} e^{-(x^2 + p^2)} L_n\left( 2(x^2 + p^2) \right)
$$

where $L_n(u)$ is the $n$-th standard Laguerre polynomial:
- $L_0(u) = 1 \implies W_0(x, p) = \frac{1}{\pi} e^{-(x^2 + p^2)}$ (Gaussian vacuum hump)
- $L_1(u) = 1 - u \implies W_1(x, p) = \frac{1}{\pi} e^{-(x^2 + p^2)} [2(x^2 + p^2) - 1]$ (Doughnut with negative crater at center)


In [ ]:
# Comprehensive Numerical Wigner Function Solver in Truncated Fock Basis

def compute_wigner(state, xvec, pvec):
    """Computes Wigner quasiprobability distribution W(x, p) on a 2D grid from state vector or density matrix."""
    if state.ndim == 1:
        psi = state / np.linalg.norm(state)
        rho = np.outer(psi, psi.conj())
    else:
        rho = state / np.real(np.trace(state))
        
    N = rho.shape[0]
    X, P = np.meshgrid(xvec, pvec)
    # Dimensionless phase-space complex coordinate alpha = (x + i p) / sqrt(2)
    alpha = (X + 1j * P) / np.sqrt(2.0)
    z = 4.0 * np.abs(alpha)**2  # = 2 * (x^2 + p^2)
    exp_factor = (1.0 / np.pi) * np.exp(-2.0 * np.abs(alpha)**2)
    
    W = np.zeros_like(X, dtype=np.complex128)
    fact = np.array([float(math.factorial(n)) for n in range(N)])
    
    # Evaluate Fock-basis Laguerre expansion:
    # W(x, p) = sum_{n, m} rho_{n, m} * W_{n, m}(x, p)
    for n in range(N):
        # Diagonal elements (n == m)
        if abs(rho[n, n]) > 1e-15:
            L_nn = genlaguerre(n, 0)(z)
            W += rho[n, n] * ((-1)**n) * L_nn
            
        # Off-diagonal elements (m > n)
        for m in range(n + 1, N):
            if abs(rho[n, m]) > 1e-15:
                d = m - n
                coeff = ((-1)**n) * np.sqrt(fact[n] / fact[m]) * ((2.0 * np.conj(alpha))**d)
                L_nm = genlaguerre(n, d)(z)
                term = coeff * L_nm
                W += rho[n, m] * term + rho[m, n] * np.conj(term)
                
    return np.real(W) * exp_factor

# Verification: Benchmark Fock states |0> and |1> at the origin
N_test = 20
x_grid = np.linspace(-4.0, 4.0, 81)
p_grid = np.linspace(-4.0, 4.0, 81)

# Fock state |0> (Vacuum)
vac_test = np.zeros(N_test, dtype=np.complex128)
vac_test[0] = 1.0
W_vac = compute_wigner(vac_test, x_grid, p_grid)

# Fock state |1> (Single Photon)
fock1_test = np.zeros(N_test, dtype=np.complex128)
fock1_test[1] = 1.0
W_fock1 = compute_wigner(fock1_test, x_grid, p_grid)

# Center values
center_idx = len(x_grid) // 2
w_vac_00 = W_vac[center_idx, center_idx]
w_fock1_00 = W_fock1[center_idx, center_idx]

# Integrals: double quadrature over phase space
dx = x_grid[1] - x_grid[0]
dp = p_grid[1] - p_grid[0]
norm_vac = np.sum(W_vac) * dx * dp
norm_fock1 = np.sum(W_fock1) * dx * dp

print("--- Wigner Numerical Solver Verification ---")
print(f"Vacuum W(0,0):       {w_vac_00:+.6f} | Analytical: {+1.0/np.pi:+.6f}")
print(f"Fock |1> W(0,0):     {w_fock1_00:+.6f} | Analytical: {-1.0/np.pi:+.6f} (Negative!)")
print(f"Vacuum Integral:     {norm_vac:.6f} | Expected: 1.000000")
print(f"Fock |1> Integral:   {norm_fock1:.6f} | Expected: 1.000000")

assert abs(w_vac_00 - 1.0/np.pi) < 1e-4, "Vacuum W(0,0) test failed!"
assert abs(w_fock1_00 - (-1.0/np.pi)) < 1e-4, "Fock |1> W(0,0) test failed!"
assert abs(norm_vac - 1.0) < 1e-3, "Vacuum normalization failed!"
assert abs(norm_fock1 - 1.0) < 1e-3, "Fock |1> normalization failed!"
print("Wigner solver benchmark passed with high numerical accuracy!")


## 3. The Marginal Distributions Theorem

One of the most remarkable theorems of quantum phase space is that the marginal projections of $W(x, p)$ correspond **exactly to physical probability densities**:

$$
\mathcal{P}(x) = \int_{-\infty}^\infty W(x, p) \, dp = |\psi(x)|^2
$$
$$
\mathcal{P}(p) = \int_{-\infty}^\infty W(x, p) \, dx = |\tilde{\psi}(p)|^2
$$

---

### Rotated Quadrature Marginals and Radon Transform

More generally, consider a generalized quadrature operator rotated by angle $\theta$:
$$
\hat{x}_\theta = \hat{x} \cos\theta + \hat{p} \sin\theta
$$
Integrating $W(x, p)$ along a line perpendicular to $\hat{x}_\theta$ yields the exact probability distribution $\mathcal{P}_\theta(x_\theta) = \langle x_\theta | \hat{\rho} | x_\theta \rangle$:

$$
\mathcal{P}_\theta(x_\theta) = \iint_{-\infty}^\infty W(x, p) \, \delta(x\cos\theta + p\sin\theta - x_\theta) \, dx \, dp
$$

This integral transform is mathematically identical to the **Radon transform** used in medical CT scans! In Notebook 2.6, we will use this exact property to perform **Quantum State Tomography**, reconstructing the entire 2D Wigner function from experimentally measured 1D homodyne histograms!


In [ ]:
# Numerical Demonstration of the Marginal Distributions Theorem

# Fock state |1>:
# Analytical position wavefunction: psi_1(x) = (2 / sqrt(pi))^(1/2) * x * exp(-x^2 / 2)
# Position density: |psi_1(x)|^2 = sqrt(2/pi) * x^2 * exp(-x^2) ... with QHO normalization:
# phi_1(x) = (1 / pi^(1/4)) * sqrt(2) * x * exp(-x^2 / 2)
# |phi_1(x)|^2 = (2 / sqrt(pi)) * x^2 * exp(-x^2)
prob_x_analytical = (2.0 / np.sqrt(np.pi)) * (x_grid**2) * np.exp(-x_grid**2)
prob_p_analytical = (2.0 / np.sqrt(np.pi)) * (p_grid**2) * np.exp(-p_grid**2)

# Numerical marginals from Wigner function
prob_x_numerical = np.sum(W_fock1, axis=0) * dp
prob_p_numerical = np.sum(W_fock1, axis=1) * dx

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Position Marginal vs. Analytical |psi(x)|^2
ax1.plot(x_grid, prob_x_analytical, 'k-', lw=2.5, label=r'Analytical $|\phi_1(x)|^2 = \frac{2}{\sqrt{\pi}}x^2 e^{-x^2}$')
ax1.plot(x_grid, prob_x_numerical, 'r--', lw=2.0, label=r'Wigner Marginal $\int W_1(x, p)\,dp$')
ax1.set_xlim(-3.5, 3.5)
ax1.set_ylim(-0.02, 0.5)
ax1.set_xlabel(r"Position Quadrature $x$")
ax1.set_ylabel(r"Probability Density $\mathcal{P}(x)$")
ax1.set_title(r"Position Marginal Distribution: $\int W(x, p)\,dp = |\psi(x)|^2$")
ax1.legend(loc='upper right', framealpha=0.9)
ax1.grid(True)

# Panel 2: Momentum Marginal vs. Analytical |psi(p)|^2
ax2.plot(p_grid, prob_p_analytical, 'k-', lw=2.5, label=r'Analytical $|\tilde{\phi}_1(p)|^2 = \frac{2}{\sqrt{\pi}}p^2 e^{-p^2}$')
ax2.plot(p_grid, prob_p_numerical, 'b--', lw=2.0, label=r'Wigner Marginal $\int W_1(x, p)\,dx$')
ax2.set_xlim(-3.5, 3.5)
ax2.set_ylim(-0.02, 0.5)
ax2.set_xlabel(r"Momentum Quadrature $p$")
ax2.set_ylabel(r"Probability Density $\mathcal{P}(p)$")
ax2.set_title(r"Momentum Marginal Distribution: $\int W(x, p)\,dx = |\tilde{\psi}(p)|^2$")
ax2.legend(loc='upper right', framealpha=0.9)
ax2.grid(True)

plt.tight_layout()
plt.show()

# Verification assertion
max_marginal_diff = np.max(np.abs(prob_x_numerical - prob_x_analytical))
print(f"Max difference between Wigner marginal and exact analytical density: {max_marginal_diff:.2e}")
assert max_marginal_diff < 5e-3, "Marginal integral verification failed!"
print("Marginal distribution theorem numerically verified!")


In [ ]:
# Multi-State Phase Space Gallery: Vacuum, Coherent, Squeezed, and Fock |1>

N_states = 30
# 1. Vacuum state |0>
psi_0 = np.zeros(N_states, dtype=np.complex128); psi_0[0] = 1.0
W_0 = compute_wigner(psi_0, x_grid, p_grid)

# 2. Coherent state |alpha> with alpha = 1.8
alpha_val = 1.8
a_ladder = np.diag(np.sqrt(np.arange(1, N_states)), k=1)
D_mat = la.expm(alpha_val * a_ladder.T.conj() - alpha_val * a_ladder)
psi_coh = D_mat @ psi_0
W_coh = compute_wigner(psi_coh, x_grid, p_grid)

# 3. Squeezed Vacuum |0, r> with r = 0.65
r_val = 0.65
S_mat = la.expm(0.5 * r_val * (a_ladder@a_ladder - a_ladder.T.conj()@a_ladder.T.conj()))
psi_sq = S_mat @ psi_0
W_sq = compute_wigner(psi_sq, x_grid, p_grid)

# 4. Single-Photon Fock state |1>
W_1 = W_fock1

# Plotting: 4-Panel 2D Contour Gallery
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

states_data = [
    (W_0, r"Vacuum State $|0\rangle$", axes[0, 0]),
    (W_coh, rf"Coherent State $|\alpha={alpha_val}\rangle$", axes[0, 1]),
    (W_sq, rf"Squeezed Vacuum ($r={r_val}$)", axes[1, 0]),
    (W_1, r"Fock State $|1\rangle$ (Non-Classical Negativity)", axes[1, 1])
]

# Universal colormap symmetric around zero
w_limit = 0.35
levels = np.linspace(-w_limit, w_limit, 50)

for W_map, title, ax in states_data:
    cf = ax.contourf(x_grid, p_grid, W_map, levels=levels, cmap='RdBu_r', extend='both')
    # Zero contour highlighted in bold black
    ax.contour(x_grid, p_grid, W_map, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
    ax.set_aspect('equal')
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_xlabel(r"Quadrature $x$")
    ax.set_ylabel(r"Quadrature $p$")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

# Shared colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(cf, cax=cbar_ax)
cbar.set_label(r"Wigner Function $W(x, p)$", rotation=270, labelpad=15)

plt.show()


> **Physical Insight: What Does Negative Wigner Quasiprobability Mean?**
> A classical probability density cannot be negative ($P \ge 0$). Why does quantum mechanics allow $W(x, p) < 0$?
>
> In 1974, **K. S. Hudson** proved a foundational theorem (*Reports on Mathematical Physics* 6, 249): **the only pure states with non-negative Wigner functions ($W(x, p) \ge 0$) are Gaussian states (vacuum, coherent, and squeezed states)!**
>
> Any pure non-Gaussian state (such as Fock state $|1\rangle$, Schrödinger cat states, or GKP quantum error-correcting codes) MUST possess regions where $W(x, p) < 0$. These negative regions are the physical embodiment of **quantum interference in phase space**. In quantum computing, Wigner negativity is a necessary resource for exponential quantum computational advantage (Gottesman-Knill theorem)!


In [ ]:
# Interactive Exploration: Phase Space Landscape Explorer

def explore_wigner_gallery(state_type='Fock State', param=1.0):
    """Interactive visualization of Wigner function contours and 1D marginal slices."""
    N_calc = 25
    x_interactive = np.linspace(-3.5, 3.5, 61)
    p_interactive = np.linspace(-3.5, 3.5, 61)
    
    if state_type == 'Fock State':
        n = int(param)
        psi = np.zeros(N_calc, dtype=np.complex128)
        if n < N_calc:
            psi[n] = 1.0
        title_str = rf"Fock State $|{n}\rangle$"
        
    elif state_type == 'Coherent State':
        alpha = param
        p_coh = np.array([(alpha**(2*n)) * np.exp(-alpha**2) / math.factorial(n) for n in range(N_calc)])
        psi = np.sqrt(p_coh).astype(np.complex128)
        title_str = rf"Coherent State $|\alpha={alpha:.1f}\rangle$"
        
    else:  # Squeezed Vacuum
        r = param
        a = np.diag(np.sqrt(np.arange(1, N_calc)), k=1)
        S = la.expm(0.5 * r * (a@a - a.T.conj()@a.T.conj()))
        vac = np.zeros(N_calc, dtype=np.complex128); vac[0] = 1.0
        psi = S @ vac
        title_str = rf"Squeezed Vacuum ($r={r:.2f}$)"
        
    W = compute_wigner(psi, x_interactive, p_interactive)
    min_w = np.min(W)
    max_w = np.max(W)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Panel 1: Contour Map
    vmax = max(0.35, max_w)
    cf = ax1.contourf(x_interactive, p_interactive, W, levels=np.linspace(-vmax, vmax, 40),
                      cmap='RdBu_r', extend='both')
    ax1.contour(x_interactive, p_interactive, W, levels=[0.0], colors='black', linewidths=1.5, linestyles='--')
    ax1.set_aspect('equal')
    ax1.set_xlabel("x")
    ax1.set_ylabel("p")
    ax1.set_title(f"{title_str} (Min W: {min_w:+.3f})")
    plt.colorbar(cf, ax=ax1, fraction=0.046, pad=0.04)
    
    # Panel 2: 1D Position Marginal Slice
    dp_val = p_interactive[1] - p_interactive[0]
    p_x = np.sum(W, axis=0) * dp_val
    ax2.plot(x_interactive, p_x, 'b-', lw=2.2, label=r'$\mathcal{P}(x) = \int W(x, p)\,dp$')
    ax2.set_xlabel("x")
    ax2.set_ylabel("Probability Density")
    ax2.set_title("Projected Position Marginal Density")
    ax2.set_xlim(-3.5, 3.5)
    ax2.grid(True)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_wigner_gallery,
             state_type=Dropdown(options=['Fock State', 'Coherent State', 'Squeezed Vacuum'], value='Fock State'),
             param=FloatSlider(min=0.0, max=4.0, step=0.5, value=1.0, description='Parameter:'))
else:
    # Static execution fallback for headless environments
    explore_wigner_gallery(state_type='Fock State', param=1.0)


## 4. Self-Check Exercises and Pedagogical Challenges

Consolidate your comprehension of Wigner quasiprobability distributions and Moyal phase space algebra.

---

### Challenge 1: Proof of Moyal's Overlap Formula
Using the definition of the Wigner function:
$$
W_\rho(x, p) = \frac{1}{\pi} \int_{-\infty}^\infty \langle x + y | \hat{\rho} | x - y \rangle e^{-2ipy} \, dy
$$
1. Prove that for any two density matrices $\hat{\rho}_1$ and $\hat{\rho}_2$:
   $$
   2\pi \iint_{-\infty}^\infty W_1(x, p) W_2(x, p) \, dx \, dp = \mathrm{Tr}(\hat{\rho}_1 \hat{\rho}_2)
   $$
2. Show that if $\hat{\rho}_1 = |n\rangle\langle n|$ and $\hat{\rho}_2 = |m\rangle\langle m|$ are orthogonal Fock states ($n \neq m$), the integral of the product of their Wigner functions vanishes identically:
   $$
   \iint_{-\infty}^\infty W_n(x, p) W_m(x, p) \, dx \, dp = 0
   $$

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
Substitute the definition of $W_1$ and $W_2$ into the double integral:
$$
I = 2\pi \iint dx \, dp \left[ \frac{1}{\pi} \int dy \, \langle x+y|\hat{\rho}_1|x-y\rangle e^{-2ipy} \right] \left[ \frac{1}{\pi} \int dz \, \langle x+z|\hat{\rho}_2|x-z\rangle e^{-2ipz} \right]
$$
Rearranging integrals:
$$
I = \frac{2}{\pi} \iiint dx \, dy \, dz \, \langle x+y|\hat{\rho}_1|x-y\rangle \langle x+z|\hat{\rho}_2|x-z\rangle \int_{-\infty}^\infty dp \, e^{-2ip(y+z)}
$$
The integral over $p$ gives a Dirac delta function:
$$
\int_{-\infty}^\infty e^{-2ip(y+z)} \, dp = \pi \delta(y + z) \quad \Longrightarrow \quad z = -y
$$
Substituting $z = -y$:
$$
I = 2 \iint dx \, dy \, \langle x+y|\hat{\rho}_1|x-y\rangle \langle x-y|\hat{\rho}_2|x+y\rangle
$$
Perform the coordinate change $u = x + y$, $v = x - y$. The Jacobian determinant is:
$$
J = \left| \frac{\partial(u, v)}{\partial(x, y)} \right| = \begin{vmatrix} 1 & 1 \\ 1 & -1 \end{vmatrix} = -2 \quad \Longrightarrow \quad dx \, dy = \frac{du \, dv}{2}
$$
Therefore:
$$
I = 2 \iint \langle u | \hat{\rho}_1 | v \rangle \langle v | \hat{\rho}_2 | u \rangle \frac{du \, dv}{2} = \int du \, \langle u | \hat{\rho}_1 \left( \int |v\rangle\langle v| \, dv \right) \hat{\rho}_2 | u \rangle
$$
Using the resolution of identity $\int |v\rangle\langle v| \, dv = \hat{I}$:
$$
I = \int du \, \langle u | \hat{\rho}_1 \hat{\rho}_2 | u \rangle = \mathrm{Tr}(\hat{\rho}_1 \hat{\rho}_2) \quad \blacksquare
$$

**Part 2 Solution:**
For Fock states, $\mathrm{Tr}(|n\rangle\langle n| \cdot |m\rangle\langle m|) = |\langle n|m\rangle|^2 = \delta_{nm}$.
For $n \neq m$, the trace is zero. Therefore:
$$
2\pi \iint W_n(x, p) W_m(x, p) \, dx \, dp = 0
$$
The Wigner functions of orthogonal quantum states are orthogonal in phase space under the standard $L^2$ inner product!
</details>

---

### Challenge 2: Numerical Verification with QuTiP
Using QuTiP, compute the Wigner function for a single-photon Fock state $|1\rangle$ and verify that the minimum value is $W_{\text{min}} = -1/\pi \approx -0.3183$.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

```python
if HAS_QUTIP:
    import qutip as qt
    N = 15
    fock1 = qt.fock(N, 1)
    x = np.linspace(-3, 3, 100)
    W_qt = qt.wigner(fock1, x, x)
    min_w = np.min(W_qt)
    print(f"QuTiP Minimum W(x, p): {min_w:.6f} (Expected: {-1.0/np.pi:.6f})")
    assert abs(min_w - (-1.0/np.pi)) < 1e-3, "QuTiP Wigner minimum test failed!"
    print("QuTiP validation confirmed Wigner negativity!")
else:
    print("QuTiP not installed; run in Google Colab to execute this test.")
```
</details>


## 5. Summary, Reference Table, and Outlook

### Comprehensive Comparison: Phase Space Wigner Signatures

| Quantum State | Wigner Form $W(x, p)$ | Center Value $W(0, 0)$ | Negative Regions? | Physical Interpretation |
| :--- | :--- | :---: | :---: | :--- |
| **Vacuum $|0\rangle$** | $\frac{1}{\pi} e^{-(x^2 + p^2)}$ | $+\frac{1}{\pi} \approx +0.318$ | **No** ($W \ge 0$) | Symmetric minimum-uncertainty Gaussian |
| **Coherent $|\alpha\rangle$** | $\frac{1}{\pi} e^{-(x - \sqrt{2}\mathrm{Re}\alpha)^2 - (p - \sqrt{2}\mathrm{Im}\alpha)^2}$ | $\frac{1}{\pi} e^{-2|\alpha|^2}$ | **No** ($W \ge 0$) | Displaced Gaussian packet |
| **Squeezed Vacuum $|0, r\rangle$**| $\frac{1}{\pi} e^{-x^2 e^{2r} - p^2 e^{-2r}}$ | $+\frac{1}{\pi}$ | **No** ($W \ge 0$) | Elongated Gaussian ellipse |
| **Thermal $\hat{\rho}_{\text{th}}$** | $\frac{1}{\pi(2\bar{n}+1)} e^{-\frac{x^2+p^2}{2\bar{n}+1}}$ | $\frac{1}{\pi(2\bar{n}+1)}$ | **No** ($W \ge 0$) | Broad classical-like thermal Gaussian |
| **Fock State $|1\rangle$** | $\frac{1}{\pi} e^{-(x^2+p^2)}[2(x^2+p^2)-1]$ | **$-\frac{1}{\pi} \approx -0.318$** | **Yes** ($W < 0$) | Doughnut with negative central crater |
| **Fock State $|n\rangle$** | $\frac{(-1)^n}{\pi} e^{-(x^2+p^2)} L_n(2(x^2+p^2))$ | $\frac{(-1)^n}{\pi}$ | **Yes** ($n \ge 1$) | $n$ alternating concentric quantum rings |

---

### Key Takeaways
1. **The Quasiprobability Bridge:** $W(x, p)$ provides an exact, invertible phase-space representation of the quantum density matrix while strictly obeying marginal probability projections.
2. **Royer Parity Principle:** The center value $W(0, 0) = \frac{1}{\pi}(P_{\text{even}} - P_{\text{odd}})$ reflects photon parity; odd Fock states are guaranteed to have negative origin values.
3. **Moyal Orthogonality:** Orthogonal quantum states have orthogonal Wigner distributions with vanishing phase-space overlap integral.

---

### Primary References
- **Wigner, E.** (1932). *On the quantum correction for thermodynamic equilibrium*. Physical Review, 40(5), 749–759.
- **Royer, A.** (1977). *Wigner function as the expectation value of a parity operator*. Physical Review A, 15(2), 449–450.
- **Hudson, R. L.** (1974). *When is the Wigner quasi-probability density non-negative?*. Reports on Mathematical Physics, 6(2), 249–252.
- **Leonhardt, U.** (1997). *Measuring the Quantum State of Light*. Cambridge University Press. Chapter 3: Wigner Function.

---

### Next in the Curriculum
In **Notebook 2.2: Wigner Negativity as a Metric for Quantum Non-Classicality**, we explore Hudson's theorem in rigorous depth, formulate the Wigner negativity volume $\delta(\hat{\rho}) = \iint (|W| - W) \, dx \, dp$, and examine its role as a computational resource in continuous-variable quantum information.
